## Output Analysis V2
Similar to the original notebook, but focuses on plotting different model runs side by side.

In [6]:
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
from model_datasets.forcing import LM4Forcing
from model_datasets.output import LM4ModelOutput
from model_datasets.snotel import PNNLSnotel

from pathlib import Path
from dotenv import load_dotenv
load_dotenv(Path.cwd().parents[1] / ".env")

True

In [2]:
# Run settings
site = 'Cumbres Trestle'
model_runs = ['s01011010_gbc_v0005','s01011010_ghelaps_v0008']
forcing_file = '01011010_v0.10.1'

#### Load Forcing

In [4]:
forcing_str, forcing_version = forcing_file.split('_')
forcing_version = forcing_version[1:]

forcing = LM4Forcing(site=site, forcing_string=forcing_str, version=forcing_version, spinup=False)

In [5]:
forcing_vars = forcing.get_variables()
forcing_vars.head()

,dimensions,long_name,units
Tair,"(time, latitude, longitude)",Near-surface air Temperature,K
Qair,"(time, latitude, longitude)",Near-surface specific humidity,kg/kg
Psurf,"(time, latitude, longitude)",Surface Pressure,Pa
Rainf,"(time, latitude, longitude)",Rainfall rate,kg/m2/s
Snowf,"(time, latitude, longitude)",Snowfall rate,kg/m2/s


### Load SNOTEL

In [7]:
import os

In [9]:
snotel = PNNLSnotel(site, os.environ['SNOTEL_PATH'])
snotel.load_data()
snotel.data.head()

,T_max_C,T_min_C,T_avg_C,precip_mm,swe_mm
Date,,,,,
1980-08-06 00:00:00-06:00,NaN,NaN,NaN,0.00,0.0
1980-08-07 00:00:00-06:00,NaN,NaN,NaN,2.54,0.0
1980-08-08 00:00:00-06:00,NaN,NaN,NaN,0.00,0.0
1980-08-09 00:00:00-06:00,NaN,NaN,NaN,0.00,0.0
1980-08-10 00:00:00-06:00,NaN,NaN,NaN,0.00,0.0


In [ ]:
snotel_data = snotel.data

# Remove timezone from index
snotel_data.index = snotel_data.index.tz_localize(None)


#### Load Models

In [ ]:
model = LM4ModelOutput(site=site, forcing_string=forcing_str, namelist=namelist, version=version, spinup=False)

### Snow
Compare modeled snow output to observed and precipitation and temperature.

In [ ]:
model_snow = model.get_data(['snow','snow_depth'],'land_daily')

In [ ]:
# Get range of dates for plotting
start_date = model_snow.index[0]
end_date = model_snow.index[-1]

In [ ]:
Tair = forcing.get_data('Tair')
Tair = Tair - 273.15  # Convert from Kelvin to Celsius
Tair_max = Tair.resample('D').max()
Tair_min = Tair.resample('D').min()

In [ ]:
Tair.head()

In [ ]:
Tair_flags = forcing.get_flags('Tair')
#Tair_flags['1992-09-30':'1992-12'].head(20)

In [ ]:
# Extract precipitation and recombine
snowfall = forcing.get_data('Snowf')
rainfall = forcing.get_data('Rainf')

precipdf = pd.DataFrame({'snowf':snowfall,'rainf':rainfall})
precipdf['combined'] = (precipdf['snowf'] + precipdf['rainf'])*3600 #Convert to hourly total kg/m2/s

In [ ]:
precip_daily = precipdf['combined'].resample('1D').sum()

In [ ]:
# Plot snow
fig, ax = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

Tair_max.plot(ax=ax[0], label='Tair (Daily Max)', color='red')
Tair_min.plot(ax=ax[0], label='Tair (Daily Min)', color='blue')
snotel_data.loc[start_date:end_date, 'T_max_C'].plot(ax=ax[0], label='SNOTEL T max', style='--r')
snotel_data.loc[start_date:end_date, 'T_min_C'].plot(ax=ax[0], label='SNOTEL T max', style='--b')

precip_daily.plot(ax=ax[1], label='Forcing Precip', color='red')
snotel_data.loc[start_date:end_date, 'precip_mm'].plot(ax=ax[1], label='SNOTEL Precip', color='blue')

model_snow['snow'].plot(ax=ax[2], label='Model SWE', color='orange')
snotel_data.loc[start_date:end_date, 'swe_mm'].plot(ax=ax[2], label='SNOTEL SWE', color='blue')

# Add a figure title based on site name
plt.suptitle(f'{site}', fontsize=16, y=0.9)
ax[0].set_ylabel('Temperature (°C)')
ax[1].set_ylabel('Precip (mm)')
ax[2].set_ylabel('SWE (mm)')
ax[0].legend()
ax[1].legend()
ax[2].legend()

In [ ]:
# Plot just showing SWE comparison, use in presentations
fig, ax = plt.subplots(figsize=(12, 6))
snotel_data.loc[:, 'swe_mm'].plot(ax=ax, label='SNOTEL SWE', color='blue')
model_snow['snow'].plot(ax=ax, label='Model SWE', color='red', alpha=0.5)
plt.title(f'SWE Comparison at {site}', fontsize=16)
ax.set_ylabel('SWE (mm)')
ax.legend()

#### RMSE Calculation

In [ ]:
# Shift snotel data index by 12 hours to match model output timing
snotel_data.index = snotel_data.index + pd.Timedelta(hours=12)

In [ ]:
# Combine model and snotel data
combined_df = pd.merge(model_snow, snotel_data[['swe_mm']], left_index=True, right_index=True, how='inner')

In [ ]:
# Calculate the difference between model and snotel SWE
combined_df['swe_diff'] = combined_df['snow'] - combined_df['swe_mm']

In [ ]:
# Calculate the RMSE, don't use first year of data to avoid spinup effects
combined_df = combined_df[combined_df.index >= (combined_df.index.min() + pd.DateOffset(years=1))]
rmse = (combined_df['swe_diff']**2).mean()**0.5
print(f'RMSE between model and SNOTEL SWE: {rmse:.2f} mm')

In [ ]:
# nRMSE
nrmse = rmse / combined_df['swe_mm'].mean()
print(f'mean SNOTEL SWE: {combined_df["swe_mm"].mean():.2f} mm')
print(f'nRMSE = {nrmse}')

### Bias

In [ ]:
bias = combined_df['swe_diff'].mean()
print(f'Bias: {bias}')

In [ ]:
nbias = bias / combined_df['swe_mm'].mean()
print(f'nBias: {nbias}')

### Snow Days Calc

In [ ]:
test = combined_df.loc['2015-10':'2016-07',['snow','swe_mm']]
print(f'SNOTEL: {test.loc[test['swe_mm'] > 5, 'swe_mm'].count()}')
print(f'Model: {test.loc[test['snow'] > 5, 'snow'].count()}')

In [ ]:
#Whole dataset
print(f'SNOTEL: {combined_df.loc[combined_df['swe_mm'] > 5, 'swe_mm'].count()}')
print(f'Model: {combined_df.loc[combined_df['snow'] > 5, 'snow'].count()}')